In [9]:
import scanpy as sc
import pandas as pd
import numpy as np
import scrublet as scr
import matplotlib.pyplot as plt
import matplotlib
from matplotlib import rcParams
from matplotlib import rc
from matplotlib import colors
import seaborn as sb
from collections import Counter
import glob

sc.settings.set_figure_params(dpi = 160, color_map = 'RdPu', dpi_save = 300, format = 'pdf')
plt.rcParams["figure.figsize"] = [6,6]
plt.rcParams["pdf.fonttype"] = 42 # add pdf text as one piece instead of letter by letter

In [10]:
sb.set_palette('colorblind')

In [11]:
# set working directory
%cd /nfs/team205/cs42/HSC/HSC_new_to_transfer/

/nfs/team205/cs42/HSC/HSC_new_to_transfer


In [12]:
# figure directory
fig_path = '/home/jovyan/mount/gdrive/HSC/chenqu_jhub/'
sc.settings.figdir = fig_path

# Download data

Data downloaded from https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE216595

In [5]:
# the processed h5ad file only contains confidently assigned TF cells
# lenti = sc.read('lenti_data/GSE216595_180124_perturb.h5ad')

In [18]:
from urllib.request import urlretrieve

DATA_DIR = 'lenti_data/'

RAW_SAMPLES = {
    f"S{i}": f"GSM{6681038 + i}_180124_perturb_S{i}" for i in range(1, 9)
}
TF_MAP_FILE = DATA_DIR + "GSM6681047_180124_TFmap.csv.gz"

tf_map = pd.read_csv(
    TF_MAP_FILE, header=None, names=["cell_barcode", "TF_ORF"],
    dtype=str, keep_default_na=False,
)
tf_map["cell_barcode"] = tf_map["cell_barcode"].str.replace(r"-1$", "", regex=True)
if tf_map["cell_barcode"].duplicated().any():
    raise ValueError("TF map contains duplicate cell barcodes.")
barcode_to_orf = tf_map.set_index("cell_barcode")["TF_ORF"]
raw_tf_orfs = sorted(set(tf_map["TF_ORF"]) - {"NA", "AMB"})
orf_to_gene = {orf: orf.split("-NM_", 1)[0] for orf in raw_tf_orfs}
raw_genes = sorted(set(orf_to_gene.values()))
print(f"TF map: {len(raw_tf_orfs)} ORF constructs representing {len(raw_genes)} genes")
print(tf_map["TF_ORF"].value_counts().loc[["NA", "AMB"]].to_string())

TF map: 87 ORF constructs representing 67 genes
TF_ORF
NA     8079
AMB    8515


In [37]:
def read_compressed_legacy_10x(prefix):
    """Read GEO's gzipped legacy 10x triplet into an AnnData object."""
    adata = sc.read_mtx(DATA_DIR + f"{prefix}_matrix.mtx.gz").T
    features = pd.read_csv(
        DATA_DIR + f"{prefix}_genes.tsv.gz", sep="\t", header=None,
        names=["gene_id", "gene"], dtype=str,
    )
    barcodes = pd.read_csv(
        DATA_DIR + f"{prefix}_barcodes.tsv.gz", sep="\t", header=None,
        names=["cell_barcode_10x"], dtype=str,
    )["cell_barcode_10x"]
    adata.var_names = features["gene"].to_numpy()
    adata.var["gene_id"] = features["gene_id"].to_numpy()
    adata.var_names_make_unique()
    adata.obs_names = barcodes.to_numpy()
    adata.obs["cell_barcode_10x"] = barcodes.to_numpy()
    return adata

# 1. Load every matrix as an AnnData object.
raw_adatas = {}
for sample, prefix in RAW_SAMPLES.items():
    raw_adatas[sample] = read_compressed_legacy_10x(prefix)
    print(f"{sample}: {raw_adatas[sample].n_obs:,} cells loaded")

# 2. Concatenate all eight samples into one AnnData object.
adata_raw_all = sc.concat(
    raw_adatas, join="inner", label="sample", index_unique="-", merge="same"
)
del raw_adatas
missing = sorted(set(raw_genes) - set(adata_raw_all.var_names))
if missing:
    raise ValueError(f"Genes missing from the concatenated object: {missing}")
print(adata_raw_all)

# Apply the same mitochondrial QC threshold used for the processed H5AD.
adata_raw_all.var["mitochondrial"] = adata_raw_all.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(
    adata_raw_all, qc_vars=["mitochondrial"], percent_top=None,
    log1p=False, inplace=True,
)
adata_raw_all.obs["percent_mito"] = (
    adata_raw_all.obs["pct_counts_mitochondrial"] / 100
)
n_before_mito_qc = adata_raw_all.n_obs
adata_raw_all = adata_raw_all[adata_raw_all.obs["percent_mito"] < 0.10].copy() # remove mit% >= 0.1 as per original paper
print(
    f"Mitochondrial QC retained {adata_raw_all.n_obs:,}/{n_before_mito_qc:,} cells "
    f"(removed {n_before_mito_qc - adata_raw_all.n_obs:,})"
)

# 3. Normalize the filtered count matrix and apply log1p.
adata_raw_all.obs["library_size"] = np.asarray(adata_raw_all.X.sum(axis=1)).ravel()
sc.pp.normalize_total(adata_raw_all, target_sum=10_000)
sc.pp.log1p(adata_raw_all)

# 4. Map the deposited TF calls onto the concatenated observations.
adata_raw_all.obs["cell_barcode"] = (
    adata_raw_all.obs["cell_barcode_10x"].astype(str)
    .str.replace(r"-1$", "", regex=True)
)
adata_raw_all.obs["TF_ORF"] = (
    adata_raw_all.obs["cell_barcode"].map(barcode_to_orf).fillna("UNMAPPED")
)

raw_expression = adata_raw_all[:, raw_genes].X.toarray().T
raw_cells = adata_raw_all.obs.copy()
print(f"Combined: {adata_raw_all.n_obs:,} cells x {adata_raw_all.n_vars:,} genes")
print(
    adata_raw_all.obs["TF_ORF"].value_counts().reindex(
        ["NA", "AMB", "UNMAPPED"], fill_value=0
    ).to_string()
)

S1: 10,000 cells loaded
S2: 10,000 cells loaded
S3: 10,000 cells loaded
S4: 10,000 cells loaded
S5: 10,000 cells loaded
S6: 10,000 cells loaded
S7: 10,000 cells loaded
S8: 10,000 cells loaded
AnnData object with n_obs × n_vars = 80000 × 33694
    obs: 'cell_barcode_10x', 'sample'
    var: 'gene_id'
Mitochondrial QC retained 77,018/80,000 cells (removed 2,982)
Combined: 77,018 cells x 33,694 genes
TF_ORF
NA          7451
AMB         8570
UNMAPPED       0


In [40]:
raw_gene_position = {gene: i for i, gene in enumerate(raw_genes)}
no_barcode_mask = raw_cells["TF_ORF"].eq("NA").to_numpy()
if no_barcode_mask.sum() < 2:
    raise ValueError("Fewer than two explicit no-barcode cells were found.")

raw_rows = []
for tf_orf in raw_tf_orfs:
    gene = orf_to_gene[tf_orf]
    values = raw_expression[raw_gene_position[gene], :]
    barcode_mask = raw_cells["TF_ORF"].eq(tf_orf).to_numpy()
    barcode_values = values[barcode_mask]
    control_values = values[no_barcode_mask]
    raw_rows.append({
        "TF_ORF": tf_orf,
        "gene": gene,
        "barcode_cells": int(barcode_mask.sum()),
        "no_barcode_cells": int(no_barcode_mask.sum()),
        "barcode_mean": barcode_values.mean() if len(barcode_values) else np.nan,
        "barcode_variance": barcode_values.var(ddof=1) if len(barcode_values) > 1 else np.nan,
        "no_barcode_mean": control_values.mean(),
        "no_barcode_variance": control_values.var(ddof=1),
        "mean_difference": (
            barcode_values.mean() - control_values.mean() if len(barcode_values) else np.nan
        ),
    })

raw_results = pd.DataFrame(raw_rows).sort_values("mean_difference", ascending=False)
RAW_RESULTS_FILE = "csv/10x_raw_87tf_vs_no_barcode_summary.csv"
raw_results.to_csv(RAW_RESULTS_FILE, index=False)